In [ ]:
library(tidyverse)
library(nycflights13)

In [ ]:
# function + across()
normalize <- function(x){
  (x-mean(x,na.rm=TRUE))/sd(x,na.rm=TRUE) #z-score
}

flights |> 
  mutate(across(c(dep_delay,arr_delay),normalize),.keep="used")

#normalize(flights$dep_delay)
#normalize(flights$arr_delay)

#apply function across the variables

In [ ]:
#function + select.
high_na <- function(x){
  mean(is.na(x)) < 0.1
}
flights |> 
  select(where(high_na))

#show me only the variables where more then 90% of the data is actually present.
mean(is.na(flights$year))
mean(is.na(flights$dep_time)) < 0.1

In [ ]:
#function + filter
is_extreme_delay <- function(x){
  x >120
}

flights |> 
  filter(is_extreme_delay(dep_delay)) |> select(flight,dep_delay)

In [ ]:
# function + group_by() + summarize()
delay_summary <- function(x) {
  tibble(
    avg=mean(x,na.rm=TRUE),
    med=median(x,na.rm=TRUE),
    max=max(x,na.rm=TRUE),
    min=min(x,na.rm=TRUE)
  )
}

flights |> 
  group_by(carrier) |> 
  summarise(delay_summary(arr_delay))

In [ ]:
flights |> 
  group_by(carrier) |> 
  summarise(avg=mean(arr_delay,na.rm=TRUE),
            med=median(arr_delay,na.rm=TRUE),
            max=max(arr_delay,na.rm=TRUE),
            min=min(arr_delay,na.rm=TRUE))

In [ ]:
#function + returns single.
safe_mean <-function(x){
  if(all(is.na(x))) return(NA)
  mean(x,na.rm=TRUE)
}

flights |> 
  group_by(carrier) |> 
  summarise(avg_delay=safe_mean(arr_delay))

In [ ]:
# function with parameters
#delay flag
delay_flag <-function(x,threshold=30){
  x > threshold
}
flights |> 
  mutate(late=delay_flag(arr_delay,45),.keep="used")

In [27]:
#function inside the case_when()
#6 and 7
is_weekend <- function(x){
  x %in% c(6,7)
}

flights |> 
  mutate(weekend_flag=case_when(
    is_weekend(day) ~"Weekend", #tild #delta 
    TRUE~"Weekday"
  ),.keep="used")

# A tibble: 336,776 × 2
     day weekend_flag
   <int> <chr>       
 1     1 Weekday     
 2     1 Weekday     
 3     1 Weekday     
 4     1 Weekday     
 5     1 Weekday     
 6     1 Weekday     
 7     1 Weekday     
 8     1 Weekday     
 9     1 Weekday     
10     1 Weekday     
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
#anonymous
flights |> 
  mutate(across(dep_delay:arr_delay,~.x/60),.keep="used")

In [32]:
#function +arrange
delay_score <-function(dep,arr){
  dep+arr # totalDelay
}

flights |> 
  arrange(delay_score(dep_delay,arr_delay)) |> select(dep_delay,arr_delay)

# A tibble: 336,776 × 2
   dep_delay arr_delay
       <dbl>     <dbl>
 1       -14       -86
 2       -16       -79
 3       -33       -58
 4       -14       -70
 5       -20       -63
 6       -15       -67
 7       -17       -65
 8       -13       -68
 9       -13       -67
10       -18       -62
# ℹ 336,766 more rows
# ℹ Use `print(n = ...)` to see more rows

In [ ]:
flights |> 
  mutate(total=delay_score(dep_delay,arr_delay),.keep="used") |> 
  arrange(total)  |> filter(!is.na(total)) |> head(100)

# A tibble: 100 × 3
   dep_delay arr_delay total
       <dbl>     <dbl> <dbl>
 1       -14       -86  -100
 2       -16       -79   -95
 3       -33       -58   -91
 4       -14       -70   -84
 5       -20       -63   -83
 6       -15       -67   -82
 7       -17       -65   -82
 8       -13       -68   -81
 9       -13       -67   -80
10       -18       -62   -80
# ℹ 90 more rows
# ℹ Use `print(n = ...)` to see more rows